# Tools and Agents (open source): Tool-Calling Basics

### Outline
- A hard-coded function and a matching JSON tool schema
- Ask the model a question that needs the tool, with tools available
- Ask the model a question that doesn't need it
- Execute the tool call ourselves and complete the round trip

Open-source recode of `03-Functions-Tools-and-Agents-with-LangChain/L1-openai_functions_student.ipynb`.
The original notebook calls the raw `openai` SDK directly (no LangChain yet) to show what
"function calling" looks like at the wire level: you send a JSON schema of your function(s)
alongside the messages, and the model either replies normally or replies with a call you're
supposed to execute yourself.

Ollama's `/api/chat` endpoint speaks the same shape (OpenAI-compatible
`tools=[{"type": "function", "function": {...}}]`), so this recodes the lesson with the raw
`ollama` client instead of LangChain - LangChain enters in L2 onward, same as the original
course.

## Setup

This notebook is the open-source / Ollama-cloud recode of the matching lesson in
[`openai_agentic_ai_course`](../../openai_agentic_ai_course/), reusing the shared
`common.py` / `tracing.py` helpers already built for
[`tools_and_agent`](../../tools_and_agent/) rather than duplicating them here.

- **Model**: `ChatOllama`, pointed at the Ollama cloud endpoint configured in the
  repo-root `.env` (`OLLAMA_MODEL` / `OLLAMA_BASE_URL` / `OLLAMA_API_KEY`).
- **Tracing**: every `.invoke()` / `.batch()` / `.stream()` call below passes
  `config=traced("run name")`, which attaches a Langfuse callback - open the
  Langfuse dashboard and filter by trace name to see this notebook's calls.
- **Kernel**: run this with the repo's `.venv` (`Python 3 (ipykernel)`) - it already
  has everything in [`requirements.txt`](../../requirements.txt) installed.

In [ ]:
import sys
from pathlib import Path

# common.py / tracing.py live in tools_and_agent/, not here - add it to sys.path
# instead of copying them, so this notebook always uses the one shared implementation.
COURSE_DIR = Path("../../tools_and_agent").resolve()
if str(COURSE_DIR) not in sys.path:
    sys.path.insert(0, str(COURSE_DIR))

import json
import os

from ollama import Client
from tracing import langfuse

There's no LangChain runnable here for Langfuse's `CallbackHandler` to attach to, so each call
is wrapped by hand in `langfuse.start_as_current_observation(..., as_type="generation")`.

In [ ]:
client = Client(
    host=os.environ["OLLAMA_BASE_URL"],
    headers={"Authorization": f"Bearer {os.environ['OLLAMA_API_KEY']}"},
)
MODEL = os.environ["OLLAMA_MODEL"]


def traced_chat(name, messages, tools=None):
    """`client.chat(...)`, logged to Langfuse as a `generation` observation."""
    with langfuse.start_as_current_observation(
        name=name,
        as_type="generation",
        model=MODEL,
        input=messages,
    ) as generation:
        response = client.chat(model=MODEL, messages=messages, tools=tools)
        generation.update(
            output=response.message.model_dump(exclude_none=True),
            usage_details={
                "input": response.prompt_eval_count,
                "output": response.eval_count,
            },
        )
        return response


def show(title, response):
    print(f"--- {title} ---")
    print("message:", response.message)

A dummy function standing in for a real backend/weather API call, plus the JSON schema that
describes it to the model - same shape as OpenAI's "functions" list, just wrapped in the
`{"type": "function", "function": {...}}` envelope Ollama's tools API expects.

In [ ]:
def get_current_weather(location, unit="fahrenheit"):
    """Get the current weather in a given location"""
    weather_info = {
        "location": location,
        "temperature": "72",
        "unit": unit,
        "forecast": ["sunny", "windy"],
    }
    return json.dumps(weather_info)


tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
                "required": ["location"],
            },
        },
    }
]

In [ ]:
messages = [{"role": "user", "content": "What's the weather like in Boston?"}]
response = traced_chat("L1: weather question, tools available", messages, tools=tools)
show("weather question, tools available", response)

A greeting doesn't need the tool - the model should reply normally even with tools available.

In [ ]:
messages = [{"role": "user", "content": "hi!"}]
response = traced_chat("L1: greeting, tools available but unused", messages, tools=tools)
show("greeting, tools available but unused", response)

# Note: unlike OpenAI's `function_call="none"` / `function_call={"name": ...}`, Ollama has no
# wire-level knob to force or forbid a tool call - the model decides on its own from the prompt
# and tool descriptions. If you need to force a specific tool, the usual trick is to only offer
# that one tool.

## Executing the call and completing the round trip

Ask again, actually run the tool the model asked for, feed the result back as a `"role": "tool"`
message, and let the model produce the final answer.

In [ ]:
messages = [{"role": "user", "content": "What's the weather like in Boston!"}]
response = traced_chat("L1: weather question again", messages, tools=tools)
show("weather question again", response)

assistant_message = response.message
messages.append(assistant_message)

In [ ]:
if assistant_message.tool_calls:
    call = assistant_message.tool_calls[0]
    args = call.function.arguments
    observation = get_current_weather(**args)
    print("tool result:", observation)

    messages.append(
        {
            "role": "tool",
            "content": observation,
            "tool_name": call.function.name,
        }
    )

    final_response = traced_chat("L1: final answer after tool result", messages)
    show("final answer after tool result", final_response)
    print()
    print(final_response.message.content)